In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

In [ ]:
BASE_DIR = Path.cwd()
PROJECT_ROOT = BASE_DIR.parent if BASE_DIR.name == "notebooks" else BASE_DIR

gi_path = os.path.join(PROJECT_ROOT, "outputs", "ihr_gi_star_2001_2025.csv")
gi_long = pd.read_csv(gi_path)  # Cell_ID, Year, Gi_star_z, Gi_star_p_sim, Gi_star_p_norm

mk_path = os.path.join(PROJECT_ROOT, "outputs", "mann_kendall", "ihr_mann_kendall_results.csv")
mk_results = pd.read_csv(mk_path)  # Cell_ID, MK_trend, MK_h, MK_p, MK_z, MK_tau, MK_s, MK_var_s, MK_slope, Test_Used

grid_path = os.path.join(PROJECT_ROOT, "data", "raw", "ihr_grid_2_5km_wgs84.gpkg")
grid = gpd.read_file(grid_path)  # Cell_ID, cell_size_km, geometry

print(f"Gi* rows: {len(gi_long):,} | MK results: {len(mk_results):,} | Grid cells: {len(grid):,}")

## Rebuild annual Hot / Not-Hot classification (no cold spots)

In [ ]:
def classify_hotspot(z, p):
    if pd.isna(z) or pd.isna(p):
        return "No Data"
    if z > 2.58 and p <= 0.01:
        return "Hot Spot 99%"
    if z > 1.96 and p <= 0.05:
        return "Hot Spot 95%"
    '''
    if z > 1.65 and p <= 0.10:
        return "Hot Spot 90%"
    return "Not Significant"'''


gi_long["Hotspot_Class"] = [
    classify_hotspot(z, p) for z, p in zip(gi_long["Gi_star_z"], gi_long["Gi_star_p_sim"])
]

# "Hot" for the annual sequence = significant at any confidence level (90/95/99%)
# gi_long["Is_Hot"] = gi_long["Hotspot_Class"].isin(["Hot Spot 90%", "Hot Spot 95%", "Hot Spot 99%"])
gi_long["Is_Hot"] = gi_long["Hotspot_Class"].isin(["Hot Spot 95%", "Hot Spot 99%"])

hot_wide = gi_long.pivot(index="Cell_ID", columns="Year", values="Is_Hot")
hot_wide = hot_wide[sorted(hot_wide.columns)]  # chronological 2001 -> 2025
years = list(hot_wide.columns)
print(f"Cells: {len(hot_wide):,} | Years: {years[0]}-{years[-1]}")

## Vectorized per-cell metrics (frequency, trailing run, prior-hot history)

In [ ]:
H = hot_wide.to_numpy(dtype=bool)  # (n_cells, 25), chronological
n_years = H.shape[1]

total_hot_years = H.sum(axis=1)
hotspot_frequency = total_hot_years / n_years
final_year_hot = H[:, -1]
ever_hot = H.any(axis=1)

# Trailing run: consecutive hot years ending at the final year
rev = H[:, ::-1]
all_true = rev.all(axis=1)
first_false = np.argmax(~rev, axis=1)  # index of first False when scanning from the end
trailing_run = np.where(all_true, n_years, first_false)

# Any hot year NOT part of the trailing run (i.e. hot activity before that final run)
hot_before_trailing_run = (total_hot_years - trailing_run) > 0

metrics_df = pd.DataFrame({
    "Cell_ID": hot_wide.index,
    "Total_Hot_Years": total_hot_years,
    "Hotspot_Frequency": hotspot_frequency,
    "Final_Year_Hot": final_year_hot,
    "Ever_Hot": ever_hot,
    "Trailing_Run_Length": trailing_run,
    "Hot_Before_Trailing_Run": hot_before_trailing_run,
})

print(metrics_df.describe(include="all"))

## Merge with Mann-Kendall results

In [ ]:
eha_df = metrics_df.merge(
    mk_results[["Cell_ID", "MK_trend", "MK_h", "MK_p", "MK_z", "MK_tau", "MK_slope"]],
    on="Cell_ID",
    how="left",
)

missing_mk = eha_df["MK_trend"].isna().sum()
if missing_mk:
    print(f"Warning: {missing_mk:,} cells have no Mann-Kendall result - check the MK notebook completed fully.")

eha_df["MK_h"] = eha_df["MK_h"].astype("boolean")  # nullable bool, preserves NaN for missing cells

## Assign EHA category

In [ ]:
gi_long["Is_Cold"] = (
    (gi_long["Gi_star_z"] < -1.96) &
    (gi_long["Gi_star_p_sim"] <= 0.05)
)

cold_wide = gi_long.pivot(
    index="Cell_ID",
    columns="Year",
    values="Is_Cold"
)

cold_wide = cold_wide[sorted(cold_wide.columns)]

C = cold_wide.to_numpy(dtype=bool)

ever_cold = C.any(axis=1)

In [ ]:
freq = eha_df["Hotspot_Frequency"].values
final_hot = eha_df["Final_Year_Hot"].values
total_hot = eha_df["Total_Hot_Years"].values
run = eha_df["Trailing_Run_Length"].values
ever_hot = eha_df["Ever_Hot"].values

mk_h = eha_df["MK_h"].fillna(False).values
mk_trend = eha_df["MK_trend"].fillna("").values


# --------------------------------------------------
# 1. NEW HOT SPOT
# Final year is hot and it has never been hot before
# --------------------------------------------------
is_new = final_hot & (total_hot == 1)

# --------------------------------------------------
# 2. CONSECUTIVE HOT SPOT
# One uninterrupted final run of hot years,
# never hot before that run, and <90% hot overall
# --------------------------------------------------
is_consecutive = (
    (run >= 2) &
    (total_hot == run) &
    (freq < 0.90)
)

# --------------------------------------------------
# 3. OSCILLATING HOT SPOT
# Final year is hot, previous cold-spot history exists,
# and <90% of years are hot
# --------------------------------------------------
is_oscillating = (
    final_hot &
    ever_cold &
    (freq < 0.90) &
    ~is_new &
    ~is_consecutive
)

# --------------------------------------------------
# 4. INTENSIFYING HOT SPOT
# >=90% of years hot, final year hot,
# and significant increasing temporal trend
# --------------------------------------------------
is_intensifying = (
    (freq >= 0.90) &
    final_hot &
    mk_h &
    (mk_trend == "increasing")
)

# --------------------------------------------------
# 5. PERSISTENT HOT SPOT
# >=90% of years hot, final year hot,
# but no significant increasing/decreasing trend
# --------------------------------------------------
is_persistent = (
    (freq >= 0.90) &
    final_hot &
    ~is_intensifying &
    ~(
        mk_h &
        (mk_trend == "decreasing")
    )
)

# --------------------------------------------------
# 6. DIMINISHING HOT SPOT
# >90% of years hot, final year hot,
# and significant decreasing temporal trend
# --------------------------------------------------
is_diminishing = (
    (freq >= 0.90) &
    final_hot &
    mk_h &
    (mk_trend == "decreasing")
)

# --------------------------------------------------
# 7. HISTORICAL HOT SPOT
# >=90% of years hot, but final year is not hot
# --------------------------------------------------
is_historical = (
    (freq >= 0.90) &
    ~final_hot
)

# --------------------------------------------------
# 8. SPORADIC HOT SPOT
# <90% hot, has been hot before,
# and has NEVER been a significant cold spot
# --------------------------------------------------
is_sporadic = (
    (freq < 0.90) &
    ever_hot &
    ~ever_cold &
    ~is_new &
    ~is_consecutive
)

# --------------------------------------------------
# FINAL CLASSIFICATION
# --------------------------------------------------
conditions = [
    is_new,
    is_consecutive,
    is_intensifying,
    is_persistent,
    is_diminishing,
    is_historical,
    is_sporadic,
    is_oscillating
]

choices = [
    "New Hot Spot",
    "Consecutive Hot Spot",
    "Intensifying Hot Spot",
    "Persistent Hot Spot",
    "Diminishing Hot Spot",
    "Historical Hot Spot",
    "Sporadic Hot Spot",
    "Oscillating Hot Spot"
]

eha_df["EHA_Category"] = np.select(
    conditions,
    choices,
    default="No Pattern Detected"
)

print(eha_df["EHA_Category"].value_counts())

## Save the full attribute CSV (join this to the grid in QGIS by Cell_ID)

In [ ]:
eha_dir = os.path.join(PROJECT_ROOT, "outputs", "eha_classification")
os.makedirs(eha_dir, exist_ok=True)

eha_csv_path = os.path.join(eha_dir, "ihr_eha_classification.csv")
eha_df.to_csv(eha_csv_path, index=False)
print(f"Saved -> {eha_csv_path}")

eha_df.head()

### Final map- EHA categories

In [ ]:
CATEGORY_COLORS = {
    "New Hot Spot":          "#ffff00",
    "Consecutive Hot Spot":  "#5b5bd6",
    "Sporadic Hot Spot":     "#ff8000",
    "Oscillating Hot Spot":  "#00a6a6",
    "Intensifying Hot Spot": "#ff0000",
    "Persistent Hot Spot":   "#800000",
    "Diminishing Hot Spot":  "#c48ca0",
    "Historical Hot Spot":   "#ffbf00",
    "No Pattern Detected":   "#e0e0e0",
}

CATEGORY_ORDER = list(CATEGORY_COLORS.keys())

plot_df = grid.merge(eha_df[["Cell_ID", "EHA_Category"]], on="Cell_ID", how="left")
plot_df["EHA_Category"] = plot_df["EHA_Category"].fillna("No Pattern Detected")
plot_df["color"] = plot_df["EHA_Category"].map(CATEGORY_COLORS)

fig, ax = plt.subplots(figsize=(12, 12))

plot_df.plot(
    color=plot_df["color"],
    linewidth=0,
    ax=ax
)

legend_handles = [
    mpatches.Patch(
        color=CATEGORY_COLORS[c],
        label=c
    )
    for c in CATEGORY_ORDER
    if c in plot_df["EHA_Category"].unique()
]

ax.legend(
    handles=legend_handles,
    loc="lower left",
    fontsize=9,
    framealpha=0.9
)

ax.set_title(
    "IHR Emerging Hot Spot Analysis (2001–2025)",
    fontsize=14
)

ax.set_axis_off()

final_map_dir = os.path.join(
    PROJECT_ROOT,
    "outputs",
    "final_map"
)

os.makedirs(final_map_dir, exist_ok=True)

final_map_path = os.path.join(
    final_map_dir,
    "ihr_eha_final_map.png"
)

fig.savefig(
    final_map_path,
    dpi=220,
    bbox_inches="tight"
)
print(f"Saved -> {final_map_path}")
plt.show()

## Export EHA results for QGIS visualization

In [ ]:

qgis_dir = os.path.join(PROJECT_ROOT, "outputs", "qgis")
os.makedirs(qgis_dir, exist_ok=True)

# Fields to carry into QGIS
qgis_cols = [
    "Cell_ID",
    "EHA_Category",
    "Total_Hot_Years",
    "Hotspot_Frequency",
    "Final_Year_Hot",
    "MK_trend",
    "MK_p",
    "MK_tau",
    "MK_slope"
]

# Keep only fields that actually exist
qgis_cols = [c for c in qgis_cols if c in eha_df.columns]

# Merge EHA attributes with grid geometry
qgis_gdf = grid.merge(
    eha_df[qgis_cols],
    on="Cell_ID",
    how="left"
)

# Export as GeoPackage
qgis_gpkg_path = os.path.join(
    qgis_dir,
    "ihr_eha_classification.gpkg"
)

qgis_gdf.to_file(
    qgis_gpkg_path,
    layer="EHA_Classification",
    driver="GPKG"
)

print(f"Saved -> {qgis_gpkg_path}")

# Quick check
print(f"\nCells exported: {len(qgis_gdf):,}")
print("\nEHA categories:")
print(qgis_gdf["EHA_Category"].value_counts())